# 项目实验总表与结论

汇总截至 Notebook 025 的结果。Notebook 016 保留最初六个固定方案的结果，供 DistilBERT 实验复现。本 notebook 读取已有结果，汇合传统模型、CNN/LSTM、DistilBERT、投票和成员消融的指标，并并列展示推理成本。

表中的验证和测试指标对应同一套清洗后的 IMDB 划分。验证集用于选择模型及权重。权重搜索的折外估计与全验证集得分分开记录，消融和折外估计的测试列留空。

In [ ]:
# 读取各阶段结果，核对旧总表与测试划分
import json

import numpy as np
import pandas as pd

from sentiment_analysis.paths import CONFIGS_DIR, TABLES_DIR

fixed_protocol = json.loads((CONFIGS_DIR / "final_test_protocol.json").read_text(encoding="utf-8"))
distilbert_protocol = json.loads((CONFIGS_DIR / "distilbert_test_protocol.json").read_text(encoding="utf-8"))
voting_protocol = json.loads((CONFIGS_DIR / "three_model_voting_test_protocol.json").read_text(encoding="utf-8"))
test_hash = fixed_protocol["files"]["test"]["sha256"]
if test_hash != distilbert_protocol["test_sha256"] or test_hash != voting_protocol["files"]["test"]["sha256"]:
    raise ValueError("The test splits differ across evaluation stages.")

historical = pd.read_csv(TABLES_DIR / "fixed_experiment_summary.csv").set_index("MODEL", verify_integrity=True)
distilbert_summary = pd.read_csv(TABLES_DIR / "distilbert_test_comparison.csv").set_index("MODEL", verify_integrity=True)
pd.testing.assert_frame_equal(distilbert_summary.loc[historical.index], historical, rtol=0, atol=1e-12)

validation_votes = pd.read_csv(TABLES_DIR / "three_model_soft_voting_validation.csv").set_index("MODEL", verify_integrity=True)
test_votes = pd.read_csv(TABLES_DIR / "three_model_voting_test_comparison.csv").set_index("MODEL", verify_integrity=True)
ablation = pd.read_csv(TABLES_DIR / "three_model_voting_ablation_validation.csv").set_index("MODEL", verify_integrity=True)

metrics = ["ACCURACY", "PRECISION", "RECALL", "F1"]
weight_columns = ["B06 WEIGHT", "TEXTCNN WEIGHT", "DISTILBERT WEIGHT"]
weighted_validation = "Weighted soft vote (same validation)"
np.testing.assert_allclose(
    validation_votes.loc[weighted_validation, weight_columns].to_numpy(dtype=float),
    voting_protocol["soft_weights"]["Weighted soft vote"], rtol=0, atol=1e-12,
)
np.testing.assert_allclose(
    validation_votes.loc[weighted_validation, metrics].to_numpy(dtype=float),
    ablation.loc["Full weighted vote", metrics].to_numpy(dtype=float), rtol=0, atol=1e-12,
)

In [ ]:
# 汇总测试方案、验证集消融与折外选权
configurations = [
    ("Three-model hard vote", "Voting comparator", "B06, TextCNN, DistilBERT; at least 2 positive votes",
     "Full validation", validation_votes.loc["Hard vote"], "Hard vote"),
    ("Three-model equal soft vote", "Voting comparator", "B06, TextCNN, DistilBERT; 1/3 each",
     "Full validation", validation_votes.loc["Equal soft vote"], "Equal soft vote"),
    ("Full weighted vote", "Primary voting candidate", "B06, TextCNN, DistilBERT; 0.2 / 0.3 / 0.5",
     "Full validation used for weight selection", validation_votes.loc[weighted_validation], "Weighted soft vote"),
    ("Weighted soft vote (out of fold)", "Weight selection check", "Weights selected separately in each fold",
     "Pooled out-of-fold validation predictions", validation_votes.loc["Weighted soft vote (out of fold)"], None),
    ("TextCNN + DistilBERT", "Voting ablation", "3/8 / 5/8",
     "Full validation", ablation.loc["Without B06"], None),
    ("B06 + DistilBERT", "Voting ablation", "2/7 / 5/7",
     "Full validation", ablation.loc["Without TextCNN"], None),
    ("B06 + TextCNN", "Voting ablation", "2/5 / 3/5",
     "Full validation", ablation.loc["Without DistilBERT"], None),
]

rows = []
for name, role, configuration, validation_basis, validation_row, test_name in configurations:
    row = {
        "MODEL": name, "ROLE": role, "CONFIGURATION": configuration,
        "VALIDATION BASIS": validation_basis,
    }
    for metric in metrics:
        row["VALIDATION " + metric] = validation_row[metric]
        row["TEST " + metric] = test_votes.loc[test_name, metric] if test_name is not None else np.nan
    rows.append(row)

existing = distilbert_summary.reset_index().assign(**{"VALIDATION BASIS": "Full validation"})
experiment_summary = pd.concat([existing, pd.DataFrame(rows)], ignore_index=True)
columns = ["MODEL", "ROLE", "CONFIGURATION", "VALIDATION BASIS"]
for metric in metrics:
    columns.extend(["VALIDATION " + metric, "TEST " + metric])
experiment_summary = experiment_summary[columns]
display(experiment_summary[["MODEL", "VALIDATION BASIS", "VALIDATION F1", "TEST F1"]].round(4))

In [ ]:
# 核对推理成本表引用的 F1，并保存项目总表
benchmark = pd.read_csv(TABLES_DIR / "inference_benchmark_comparison.csv")
score_columns = ["VALIDATION F1", "TEST F1"]
expected = experiment_summary.set_index("MODEL", verify_integrity=True).loc[benchmark["MODEL"], score_columns]
np.testing.assert_allclose(benchmark[score_columns].to_numpy(), expected.to_numpy(), rtol=0, atol=1e-12)

costs = pd.concat([
    benchmark.loc[benchmark["BATCH SIZE"] == 1].set_index("MODEL")["MEDIAN REQUEST LATENCY (ms)"],
    benchmark.loc[benchmark["BATCH SIZE"] == 16].set_index("MODEL")["THROUGHPUT (reviews/s)"],
], axis=1)
display(costs.round(2))

output_path = TABLES_DIR / "project_experiment_summary.csv"
experiment_summary.to_csv(output_path, index=False)
print(f"Summary saved: {output_path}")

## 项目结论

- **传统基线仍然有竞争力。** B06 的测试 F1 为 89.01%，高于 TextCNN 的 85.91%、单向 LSTM 的 84.03% 和双向 LSTM 的 85.16%。两种 CNN/LSTM 等权融合都比各自的单模型组合成员得分高，测试 F1 分别为 87.04% 和 87.84%。DistilBERT 以 90.90% 成为本项目测试得分最高的单模型。
- **概率投票带来了大部分融合收益。** B06、TextCNN、DistilBERT 的硬投票、等权软投票和固定加权软投票的测试 F1 依次为 91.45%、92.07% 和 92.25%。从硬投票改为等权软投票增加 0.62 个百分点，加权再增加 0.18 个百分点。加权方案相对硬投票纠正 504 条并新增 311 条错误，净少错 193 条。验证集用于选权，完整验证集 F1 为 92.87%，每折重新选权的折外 F1 为 92.71%。
- **三模型的成员贡献和 Dropout 都有验证集证据。** 按原权重比例去掉一个模型后，三组双模型的 F1 都下降，去掉 DistilBERT 时降幅最大。去掉 B06 得到的 TextCNN + DistilBERT 是其中得分最高的组合，验证 F1 为 92.17%。去掉 TextCNN 后降到 91.59%，低于三模型硬投票。TextCNN 的 Dropout=0.5 在两组配对种子下分别提高验证 F1 约 0.23 和 0.57 个百分点，结果对应以最低验证损失选择检查点的规则。
- **三模型以更高的推理开销换取最高测试 F1。** 相比 DistilBERT 单模型，三模型的测试 F1 高 1.34 个百分点，单条中位延迟从 7.32 ms 增至 9.79 ms，16 条一批的吞吐量从 582 降至 432 条/s。B06 的单条中位延迟为 0.59 ms，测试 F1 为 89.01%，提供了一个 CPU 即可运行的对照。

测试结果来自同一份去重后的 24,678 条 IMDB 评论。各阶段的测试结果陆续被查看，因此后续验证集探索与固定配置的测试结果应分别解读。模型间同时存在外部预训练、分词和输入长度等差异，DistilBERT 也只完成了一次固定种子的微调。耗时来自同一台 i9-13900HX 和 RTX 4060 Laptop GPU 机器，融合顺序执行各模型，结果取决于设备和批大小。